# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [4]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is associated with three projects in the dataset. Other domains like "Developer Tools / DevEx" and "Creative / Design / Media" also appear multiple times, but "Healthcare / MedTech" is the most frequently mentioned.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is at least one use case related to security. Specifically, the project titled "WealthifyAI 16" involves a federated learning toolkit that improves privacy in healthcare applications, which is related to security and privacy concerns. Additionally, "Pathfinder 24" mentions a secondary domain of Security, indicating relevance to security use cases.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally had positive comments about the fintech-related projects, highlighting their technical quality and real-world impact. For example, the project "GreenPulse," which falls under the Writing & Content domain with a secondary focus in FinTech, was described as "Technically ambitious and well-executed" with a high judge score of 8.9. Another project, "SkyForge," also related to FinTech, was noted as "A clever solution with measurable environmental benefit," earning a judge score of 8.4.\n\nOverall, the judges appreciated the innovative approaches and robustness of the fintech projects, though specific comments varied.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the project domains mentioned include Productivity Assistants, E-commerce / Marketplaces, Healthcare / MedTech, and Finance / FinTech. Since the context only provides a few examples and does not specify the total counts for each domain, I cannot determine definitively which is the most common. However, among the examples listed, Finance / FinTech appears twice, suggesting it might be a prominent domain in this dataset.\n\nIf you need a precise answer, additional data or a full analysis of all entries would be necessary.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "SecureNest 49" falls under the domain of E-commerce / Marketplaces and the secondary domain of Legal / Compliance. Its description indicates that it is a document summarization and retrieval system for enterprise knowledge bases, which is relevant to security and compliance considerations.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had the following comments about the fintech projects:\n\n- For the project "SynthMind" in the Finance / FinTech domain, the judges said, "Conceptually strong but results need more benchmarking."'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

Example Query: “SOC2 encryption requirements”

Why BM25 is better: BM25 focuses on exact keyword matches like “SOC2” and “encryption,” avoiding confusion with loosely related terms.

Real Scenario: If one document says “SOC2 requires encryption at rest” and another says “General data security involves encryption,” BM25 correctly picks the SOC2-specific one, while embeddings might rank the general one higher.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the project domains mentioned are Security, Creative / Design / Media, and Productivity Assistants. Since only a few examples are given, it is difficult to determine the most common domain with certainty. However, among these, Security appears to be one of the listed domains. \n\nIf I had to make an inference from the limited information, I would say that the most common project domain cannot be definitively identified from the provided data. \n\nPlease let me know if you'd like me to analyze additional data or assist further!"

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases related to security outside of federated learning for privacy improvement in healthcare applications. The use cases mentioned focus on privacy enhancement in healthcare, customer support, and finance/FinTech, but do not specifically mention security-related applications.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally gave positive remarks about the fintech projects. While specific comments about the fintech project "Pathfinder 27" are not provided, the overall evaluations highlight excellent code quality and impressive real-world impact. For example, the project received a high judge score of 9.8 out of 10 and was praised for the quality of code and use of open-source libraries. Additionally, judges noted strong quantitative results across the projects, with suggestions to include more qualitative analysis in future submissions.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," which is listed multiple times among the projects.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project titled "LearnWise 39" involves an AI model compression suite enabling on-device reasoning for IoT sensors, which is relevant to security in terms of protecting data and ensuring secure device operations.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally praised the fintech projects for their conceptual strength, strong code quality, and real-world potential. For example, one project in the fintech domain, EchoLens, received positive comments noting it was "conceptually strong but results need more benchmarking," with a high judge score of 9.0. Another fintech project, DataWeave, was recognized for "excellent code quality and use of open-source libraries," and scored a remarkable 9.8. Overall, judges noted that some fintech projects demonstrated solid work, impressive impact, and promising potential for commercialization, while also indicating areas like benchmarking and validation that could be improved.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

Generating multiple reformulations of a user query helps improve recall by searching in different ways for the same idea. It’s like having several people describe the same question using different words. This increases the chances of finding documents that use varied terms or phrasing. For example, the query “machine learning algorithms” might also be written as “ML models,” “AI techniques,” or “predictive analytics.” Each version can retrieve different relevant documents, and combining them gives a more complete set of results.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times in the examples. However, with only this limited sample, it\'s difficult to determine definitively if it is the most common overall. If you need an exact answer, more comprehensive data analysis would be necessary.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific usecases related to security mentioned. The projects mostly focus on federated learning for privacy in healthcare applications, but none of the descriptions explicitly reference security issues or solutions.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects generally highlighted their technical quality and innovative approaches. Specifically, for the projects related to finance and FinTech, the judges mentioned that one project was "comprehensive and technically mature," another described a solution as "clever with measurable environmental benefit," and another noted it as "technically ambitious and well-executed."'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," which is mentioned multiple times in the sample.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there were use cases related to security. One example is the project titled "SecureNest," which is a document summarization and retrieval system for enterprise knowledge bases, and another is "Neural Canvas," which is a low-latency inference system for multimodal agents in autonomous systems.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive remarks about some of the fintech projects. For example, the project "SecureNest," which is a document summarization and retrieval system for enterprise knowledge bases in the legal and compliance domain, received the comment "Comprehensive and technically mature approach," with a high judge score of 9.2. Another project, "DocuCheck," an AI-powered platform optimizing logistics routes for sustainability in the finance/fintech sector, was noted for being "Conceptually strong but results need more benchmarking," and received a judge score of 9.6. Overall, judges appreciated the technical maturity and innovative aspects of the fintech projects.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Legal / Compliance," with at least two projects ("TaskFlow 13" and "ChatBridge 9") falling under this category. Other domains like "Developer Tools / DevEx" and "Healthcare / MedTech" also appear multiple times, but based on the information available, "Legal / Compliance" seems to be the most frequent.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, there are projects like "BioForge" which is a medical imaging solution improving early diagnosis through vision transformers, and "Neural Canvas," which is a low-latency inference system for multimodal agents in autonomous systems, listed under the Security domain.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various opinions about the fintech projects. For example, they described one project as "technically ambitious and well-executed," another as having a "comprehensive and technically mature approach," and another found it to be "a forward-looking idea with solid supporting data." Overall, the comments highlight that the judges viewed some fintech projects as ambitious, well-structured, and promising, with positive assessments of their technical quality and potential impact.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
When FAQ sentences are short and repeat a lot, semantic chunking can get confused because all the sentences look very similar. It might split chunks in weird places, even breaking up a question and its answer. To fix this, you can use fixed-size chunks with some overlap or set rules to keep each question and answer together, so the chunks make more sense.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
# =============================================================================
# STEP 1: LOAD SYNTHETIC DATA AND SETUP
# =============================================================================

import pandas as pd
import os
import time
from uuid import uuid4
from operator import itemgetter

# Load your existing synthetic dataset (s09 assignment data)
dataset = pd.read_csv('test_dataset.csv')
print(f"✅ Loaded {len(dataset)} questions from test_dataset.csv")
print(f"📋 Columns: {list(dataset.columns)}")

# Display sample questions
print("\n📋 Sample Questions:")
for i in range(3):
    print(f"\n{i+1}. {dataset.iloc[i]['user_input']}")
    print(f"   Answer: {dataset.iloc[i]['reference'][:100]}...")
    print(f"   Contexts: {len(dataset.iloc[i]['reference_contexts'])} contexts")

✅ Loaded 4 questions from test_dataset.csv
📋 Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

📋 Sample Questions:

1. According to the analysis presented by Handa et al., how does the rapid diffusion and usage of ChatGPT reflect on societal impacts and what are the key considerations highlighted in their study?
   Answer: Handa et al. study consumer usage of ChatGPT, the first mass-market chatbot, and likely the largest,...
   Contexts: 3825 contexts

2. US mean what?
   Answer: The context discusses ChatGPT message usage in the US, including daily message counts, categories of...
   Contexts: 4051 contexts

3. Can you tell me what SOC2 codes 11 means in context of ChatGPT usage?
   Answer: Variation by Occupation Figure 23 presents variation in ChatGPT usage by user occupation, including ...
   Contexts: 3898 contexts


In [59]:
# =============================================================================
# COMPLETE ADVANCED RETRIEVAL SETUP
# =============================================================================

from advanced_retrieval_setup import main_setup

# Run complete setup with new dataset name to avoid duplicates
setup_results = main_setup(dataset, dataset_name="advancedRetrieval")

# Extract components for use in notebook
client = setup_results['client']
langsmith_dataset = setup_results['langsmith_dataset']
pdf_docs = setup_results['pdf_docs']
pdf_embeddings = setup_results['pdf_embeddings']
chat_model = setup_results['chat_model']
child_splitter = setup_results['child_splitter']
retrievers = setup_results['retrievers']
chains = setup_results['chains']
rag_prompt = setup_results['rag_prompt']

# Extract individual retrievers and chains
pdf_naive_retriever = retrievers['naive']
pdf_bm25_retriever = retrievers['bm25']
pdf_compression_retriever = retrievers['compression']
pdf_multi_query_retriever = retrievers['multi_query']
pdf_parent_retriever = retrievers['parent']
pdf_ensemble_retriever = retrievers['ensemble']
pdf_semantic_retriever = retrievers['semantic']
pdf_vectorstore = retrievers['vectorstore']

pdf_naive_retriever_chain = chains['naive_chain']
pdf_bm25_retriever_chain = chains['bm25_chain']
pdf_compression_retriever_chain = chains['compression_chain']
pdf_multi_query_retriever_chain = chains['multi_query_chain']
pdf_parent_retriever_chain = chains['parent_chain']
pdf_ensemble_retriever_chain = chains['ensemble_chain']
pdf_semantic_retriever_chain = chains['semantic_chain']

🚀 Starting Advanced Retrieval Setup...
📂 Created new LangSmith dataset: advancedRetrieval
✅ LangSmith setup complete
📄 Loading PDF data...
✅ Loaded 64 documents from PDF
✅ Components initialized
🔧 Creating all retrieval strategies...
✅ All retrievers created
🔗 Creating RAG template and chains...
✅ RAG chains created for all retrievers
🎉 Advanced Retrieval Setup Complete!


In [63]:
# =============================================================================
# STEP 8: RUN LANGSMITH EVALUATIONS
# =============================================================================

# Reload the module to get latest changes
import importlib
import evaluation_analysis
importlib.reload(evaluation_analysis)

from evaluation_analysis import run_langsmith_evaluations_only

# Run LangSmith evaluations
evaluator_llm, custom_run_config, langsmith_results = run_langsmith_evaluations_only(
    chains, dataset, "advancedRetrieval"
)

🚀 Starting LangSmith Evaluations...
✅ RAGAS evaluation setup complete
🚀 Running LangSmith evaluations...
Running Naive Retriever...
View the evaluation results for experiment: 'naive-retriever-9f462fc4' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/4a252494-a493-4b73-bcd5-f1dc337d027f/compare?selectedSessions=2dbf3175-9031-434e-bcdc-44d140701d33




4it [00:26,  6.53s/it]


Naive Retriever completed!

Running BM25 Retriever...
View the evaluation results for experiment: 'bm25-retriever-c9d0092e' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/4a252494-a493-4b73-bcd5-f1dc337d027f/compare?selectedSessions=72bf9cca-5386-488a-a637-7c2177803878




4it [00:14,  3.56s/it]


BM25 Retriever completed!

Running Compression Retriever...
View the evaluation results for experiment: 'compression-retriever-860eac1f' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/4a252494-a493-4b73-bcd5-f1dc337d027f/compare?selectedSessions=9bf16631-2bb0-4929-a16f-b53363ffbb69




...

Parent Retriever completed!

Running Ensemble Retriever...
View the evaluation results for experiment: 'ensemble-retriever-12becf91' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/4a252494-a493-4b73-bcd5-f1dc337d027f/compare?selectedSessions=91df43d7-7fcb-4133-bde4-27da59787b4e




4it [01:07, 16.87s/it]


Ensemble Retriever completed!

Running Semantic Chunk Retriever...
View the evaluation results for experiment: 'semantic-retriever-fe3178ae' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/4a252494-a493-4b73-bcd5-f1dc337d027f/compare?selectedSessions=79c9da41-43d2-424e-acc5-7bb4d70aaad8




4it [00:19,  4.81s/it]


Semantic Chunk Retriever completed!

✅ All LangSmith evaluations completed
✅ LangSmith evaluations completed


In [64]:
# =============================================================================
# STEP 9: RUN RAGAS ANALYSIS
# =============================================================================

from evaluation_analysis import run_ragas_analysis_only

# Run RAGAS analysis
ragas_scores = run_ragas_analysis_only(evaluator_llm, custom_run_config)

🔍 Starting RAGAS Analysis...
🔍 Running RAGAS analysis...

🔍 Processing Naive Retriever with 4 results...
Evaluating Naive Retriever...
📊 Sample data for Naive Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 10
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [06:37<00:00, 16.58s/it]


✅ Naive Retriever completed!

🔍 Processing BM25 Retriever with 4 results...
Evaluating BM25 Retriever...
📊 Sample data for BM25 Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 4
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [05:01<00:00, 12.57s/it]


✅ BM25 Retriever completed!

🔍 Processing Compression Retriever with 4 results...
Evaluating Compression Retriever...
📊 Sample data for Compression Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 3
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


...

✅ Parent Retriever completed!

🔍 Processing Ensemble Retriever with 4 results...
Evaluating Ensemble Retriever...
📊 Sample data for Ensemble Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 15
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [07:53<00:00, 19.71s/it]


✅ Ensemble Retriever completed!

🔍 Processing Semantic Chunk Retriever with 4 results...
Evaluating Semantic Chunk Retriever...
📊 Sample data for Semantic Chunk Retriever:
  Question: How does Writing relate to ChatGPT usage?...
  Retrieved contexts count: 10
  Reference: Writing is by far the most common work use, accoun...
  Response: Writing is a significant aspect of ChatGPT usage, ...


Evaluating: 100%|██████████| 24/24 [05:24<00:00, 13.51s/it]


✅ Semantic Chunk Retriever completed!
✅ RAGAS analysis completed
✅ RAGAS analysis completed


In [65]:
# =============================================================================
# STEP 10: GENERATE RESULTS TABLES AND FINAL SUMMARY
# =============================================================================

from evaluation_analysis import generate_results_and_summary_only

# Generate results tables and final summary
aggregated_scores, metric_columns, all_results = generate_results_and_summary_only(ragas_scores)

📊 Generating Results and Summary...
📊 Generating results tables...
📊 TABLE 1: Average Scores per Retriever
retriever                     Naive Retriever  BM25 Retriever  \
context_precision                      0.7407          0.6458   
context_recall                         1.0000          0.8750   
context_entity_recall                  0.1267          0.2765   
faithfulness                           0.7569          0.9092   
factual_correctness(mode=f1)           0.3925          0.3175   
answer_relevancy                       0.7223          0.7229   

retriever                     Compression Retriever  Multi-Query Retriever  \
context_precision                            0.7500                 0.7113   
context_recall                               0.7917                 1.0000   
context_entity_recall                        0.1807                 0.1539   
faithfulness                                 0.7530                 0.9457   
factual_correctness(mode=f1)                 0.

In [70]:
# =============================================================================
# STEP 10: GENERATE RESULTS TABLES
# =============================================================================

# Reload the module to get latest changes
import importlib
import evaluation_analysis
importlib.reload(evaluation_analysis)

from evaluation_analysis import generate_results_tables_only

# Generate results tables only
aggregated_scores, metric_columns, all_results = generate_results_tables_only(ragas_scores)

📊 Generating Results Tables...
📊 Generating results tables...
📊 TABLE 1: Average Scores per Retriever
retriever                     Naive Retriever  BM25 Retriever  \
context_precision                      0.7407          0.6458   
context_recall                         1.0000          0.8750   
context_entity_recall                  0.1267          0.2765   
faithfulness                           0.7569          0.9092   
factual_correctness(mode=f1)           0.3925          0.3175   
answer_relevancy                       0.7223          0.7229   

retriever                     Compression Retriever  Multi-Query Retriever  \
context_precision                            0.7500                 0.7113   
context_recall                               0.7917                 1.0000   
context_entity_recall                        0.1807                 0.1539   
faithfulness                                 0.7530                 0.9457   
factual_correctness(mode=f1)                 0.3025 

In [71]:
# =============================================================================
# STEP 11: GENERATE FINAL SUMMARY
# =============================================================================

from evaluation_analysis import generate_final_summary_only

# Generate final summary only
generate_final_summary_only(aggregated_scores, metric_columns)

📊 Generating Final Summary...
🏆 FINAL SUMMARY AND RECOMMENDATIONS

📊 OVERALL PERFORMANCE RANKING:
----------------------------------------
 1. context_recall: 0.9345
 2. faithfulness: 0.8633
 3. answer_relevancy: 0.7214
 4. context_precision: 0.7063
 5. factual_correctness(mode=f1): 0.3739
 6. context_entity_recall: 0.2145

🏆 BEST RETRIEVER BY METRIC:
----------------------------------------
context_precision: Compression Retriever (0.7500)
context_recall: Naive Retriever (1.0000)
context_entity_recall: Ensemble Retriever (0.3038)
faithfulness: Parent Retriever (0.9886)
factual_correctness(mode=f1): Ensemble Retriever (0.4400)
answer_relevancy: BM25 Retriever (0.7229)

🥇 OVERALL BEST RETRIEVER: Ensemble Retriever
📈 Average Score: 0.6890

✅ Activity 1 - Advanced Retrieval Evaluation Completed!
📁 Results saved and analysis complete
🔗 Check LangSmith dashboard for detailed traces: https://smith.langchain.com/
✅ Final summary completed


### Complete Performance Analysis: Quality, Cost & Latency

#### 🏆 Overall Performance Ranking

| Rank | Retriever | Avg Score | Latency (s) | Cost ($) | Tokens | Performance Tier |
|------|-----------|-----------|-------------|----------|--------|------------------|
| 1 | **Ensemble** | **0.6890** | 16.42 | $0.0091 | 58,386 | 🥇 Best Quality |
| 2 | Parent | 0.6863 | 4.37 | $0.0020 | 12,018 | 🥈 Best Value |
| 3 | BM25 | 0.6829 | 2.38 | $0.0023 | 13,733 | 🥉 Best Speed |
| 4 | Semantic | 0.6785 | 3.88 | $0.0032 | 19,486 | 🟢 Good Balance |
| 5 | Naive | 0.6771 | 4.22 | $0.0030 | 20,546 | 🟢 Good Balance |
| 6 | Compression | 0.6750 | 10.85 | $0.0021 | 11,689 | 🟡 Slow but Efficient |
| 7 | Multi-Query | 0.6714 | 11.44 | $0.0071 | 45,347 | 🔴 Expensive |

#### 📊 Performance Metrics Breakdown

##### 🎯 **Quality Scores (RAGAS)**
- **Ensemble**: 0.6890 (Best overall)
- **Parent**: 0.6863 (Excellent)
- **BM25**: 0.6829 (Very good)
- **Semantic**: 0.6785 (Good)
- **Naive**: 0.6771 (Good)
- **Compression**: 0.6750 (Acceptable)
- **Multi-Query**: 0.6714 (Acceptable)

##### ⚡ **Latency Performance**
- **Fastest**: BM25 (2.38s) - 6.9x faster than Ensemble
- **Fast**: Semantic (3.88s), Naive (4.22s), Parent (4.37s)
- **Slow**: Compression (10.85s), Multi-Query (11.44s)
- **Slowest**: Ensemble (16.42s)

##### 💰 **Cost Analysis**
- **Cheapest**: Parent ($0.0020) - 4.6x cheaper than Ensemble
- **Low Cost**: Compression ($0.0021), BM25 ($0.0023)
- **Medium Cost**: Naive ($0.0030), Semantic ($0.0032)
- **Expensive**: Multi-Query ($0.0071), Ensemble ($0.0091)

##### 📈 **Efficiency Ratios**
- **Speed Range**: 2.38s - 16.42s (6.9x difference)
- **Cost Range**: $0.0020 - $0.0091 (4.6x difference)
- **Token Range**: 11,689 - 58,386 (5.0x difference)

#### 🎯 **Best by Category**

| Category | Winner | Value | Runner-up |
|----------|--------|-------|-----------|
| **Overall Quality** | Ensemble | 0.6890 | Parent (0.6863) |
| **Speed** | BM25 | 2.38s | Semantic (3.88s) |
| **Cost Efficiency** | Parent | $0.0020 | Compression ($0.0021) |
| **Token Efficiency** | Compression | 11,689 | Parent (12,018) |
| **Best Value** | Parent | 0.6863 score, 4.37s, $0.0020 | BM25 (0.6829, 2.38s, $0.0023) |

#### 🚀 **Production Recommendations**

##### **Real-time Applications**
- **Primary**: BM25 (2.38s, $0.0023, 0.6829 score)
- **Alternative**: Semantic (3.88s, $0.0032, 0.6785 score)

##### **Cost-Sensitive Applications**
- **Primary**: Parent (4.37s, $0.0020, 0.6863 score)
- **Alternative**: Compression (10.85s, $0.0021, 0.6750 score)

##### **Quality-Critical Applications**
- **Primary**: Ensemble (16.42s, $0.0091, 0.6890 score)
- **Alternative**: Parent (4.37s, $0.0020, 0.6863 score)

##### **Balanced Performance**
- **Primary**: Parent (Best overall value)
- **Alternative**: BM25 (Best speed with good quality)

#### ⚠️ **Avoid for Production**
- **Multi-Query**: Expensive ($0.0071) and slow (11.44s)
- **Ensemble**: Extremely slow (16.42s) and expensive ($0.0091)

#### 🎯 **Final Recommendation**
**Parent Retriever** offers the optimal balance: 2nd best quality (0.6863), reasonable speed (4.37s), and lowest cost ($0.0020). For speed-critical applications, use **BM25**. For quality-critical applications, use **Ensemble**.